# Single Agent Pipeline Project

## What this project does

This notebook builds a small single agent smart assistant. The idea is simple. A user types a query, the agent works out what kind of request it is, and then it either calls the right tool or answers on its own.

The assistant handles three kinds of requests:

* Math questions, which go to a calculator tool
* Keyword requests, which go to a keyword extraction tool
* Anything else, which gets a plain general reply

Every response comes back as a small dictionary in a clean JSON style, so the output stays easy to read and easy to reuse in other code.

## What is implemented here

* The routing logic that decides where a query should go
* Integration with the calculator and keyword tools
* Basic error handling so a bad input does not crash the agent
* A little logging and one extra tool as a bonus


## The tools

Each tool does one job and nothing else. The agent decides which one to call.


In [1]:
# Tool 1: Calculator
import re

def calculator(expression):
    """Work out a basic math expression and return the answer as text."""
    # only allow numbers, spaces and simple math symbols so eval stays safe
    if not re.fullmatch(r"[0-9+*/(). %-]+", expression.strip()):
        return "Error in calculation"
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"


In [2]:
# Tool 2: Keyword Extractor

def extract_keywords(text):
    """Pull out a few of the longer, more meaningful words from the text."""
    words = text.split()
    keywords = []
    for w in words:
        clean = w.lower().strip(".,!?;:")
        if len(clean) > 4 and clean not in keywords:
            keywords.append(clean)
    return keywords[:5]


In [3]:
# Tool 3 (bonus): Word Counter

def word_count(text):
    """Count how many words are in the text."""
    return len(text.split())


## The agent logic

The agent lowercases the query and then checks it against a few simple rules in order:

* If the query mentions "calculate", it sends the math part to the calculator
* If the query mentions "keywords", it sends the text to the keyword extractor
* If the query mentions "count", it counts the words using the bonus tool
* If none of these match, it falls back to a general response

The first rule that matches wins, and every branch returns the same tidy dictionary shape. A small logger prints which route was taken, which makes it easier to follow what the agent is doing.


In [4]:
# Agent function

import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")
logger = logging.getLogger("agent")


def agent(query):
    query_lower = query.lower()

    # math questions go to the calculator
    if "calculate" in query_lower:
        logger.info("Routing to calculator")
        expression = query_lower.replace("calculate", "").strip()
        result = calculator(expression)
        if result == "Error in calculation":
            return {"type": "error", "result": result}
        return {"type": "calculation", "result": result}

    # keyword requests go to the keyword extractor
    elif "keywords" in query_lower:
        logger.info("Routing to keyword extractor")
        # drop the instruction part and keep the actual text after 'from'
        if "from" in query_lower:
            idx = query_lower.index("from") + len("from")
            text = query[idx:].strip()
        else:
            text = query
        result = extract_keywords(text)
        return {"type": "keywords", "result": result}

    # word counting as a bonus tool
    elif "count" in query_lower:
        logger.info("Routing to word counter")
        result = word_count(query)
        return {"type": "word_count", "result": result}

    # everything else gets a direct general reply
    else:
        logger.info("No tool matched, sending a general response")
        return {
            "type": "general",
            "result": "This looks like a general question. I do not have a dedicated tool for it, so here is a direct reply based on your query."
        }


## Expected output format

Every call to the agent returns a dictionary with two keys, a type and a result:

```
{
  "type": "calculation, keywords, word_count, general or error",
  "result": "the actual answer"
}
```


In [5]:
# Test cases

queries = [
    "Calculate 20 + 5",
    "Calculate 100 / 0",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "Count the words in this sentence please",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("=" * 50)


Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
Query: Calculate 100 / 0
Response: {'type': 'error', 'result': 'Error in calculation'}
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'intelligence', 'transforming', 'industries']}
Query: Count the words in this sentence please
Response: {'type': 'word_count', 'result': 7}
Query: What is machine learning?
Response: {'type': 'general', 'result': 'This looks like a general question. I do not have a dedicated tool for it, so here is a direct reply based on your query.'}


In [6]:
# Interactive mode
# Run this cell and type your own queries. Type 'exit' to stop.

while True:
    user_input = input("Enter a query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))


Enter a query (type 'exit' to stop): Calculate 45 * 12
Response: {'type': 'calculation', 'result': '540'}
Enter a query (type 'exit' to stop): Extract keywords from Cyber security protects networks from digital attacks
Response: {'type': 'keywords', 'result': ['cyber', 'security', 'protects', 'networks', 'digital']}
Enter a query (type 'exit' to stop): Count how many words are in this sentence
Response: {'type': 'word_count', 'result': 8}
Enter a query (type 'exit' to stop): Who created the Python language
Response: {'type': 'general', 'result': 'This looks like a general question. I do not have a dedicated tool for it, so here is a direct reply based on your query.'}
Enter a query (type 'exit' to stop): Calculate 10 / 0
Response: {'type': 'error', 'result': 'Error in calculation'}
Enter a query (type 'exit' to stop): exit
